# 13. N-grams in Natural Language Processing

## Learning Objectives

By the end of this notebook, you will be able to:
- Explain unigrams, bigrams, and trigrams.
- Generate n-grams from tokenized text using NLTK.
- Count n-gram frequencies.
- Compare word frequencies with n-gram frequencies.
- Understand common uses and limitations of n-grams.

## 1. What Is an N-gram?

An **n-gram** is a sequence of `n` consecutive items, usually words or tokens, in a text.

- **Unigram (n = 1):** one token at a time.
- **Bigram (n = 2):** two consecutive tokens.
- **Trigram (n = 3):** three consecutive tokens.

N-grams help represent local word order. For example, `machine learning` and `learning machine` contain the same words but are different bigrams.

In [1]:
text = "Natural language processing helps computers understand human language"
tokens = text.lower().split()

print(tokens)

['natural', 'language', 'processing', 'helps', 'computers', 'understand', 'human', 'language']


## 2. Generate N-grams with NLTK

NLTK provides `ngrams()` to generate consecutive sequences from a list of tokens. The input must be tokenized first.

In [2]:
from nltk.util import ngrams

unigrams = list(ngrams(tokens, 1))
bigrams = list(ngrams(tokens, 2))
trigrams = list(ngrams(tokens, 3))

print("Unigrams:", unigrams)
print("Bigrams:", bigrams)
print("Trigrams:", trigrams)

Unigrams: [('natural',), ('language',), ('processing',), ('helps',), ('computers',), ('understand',), ('human',), ('language',)]
Bigrams: [('natural', 'language'), ('language', 'processing'), ('processing', 'helps'), ('helps', 'computers'), ('computers', 'understand'), ('understand', 'human'), ('human', 'language')]
Trigrams: [('natural', 'language', 'processing'), ('language', 'processing', 'helps'), ('processing', 'helps', 'computers'), ('helps', 'computers', 'understand'), ('computers', 'understand', 'human'), ('understand', 'human', 'language')]


Each n-gram is returned as a tuple. For example, `('natural', 'language')` is a bigram. Converting the result to a list makes it easy to inspect or count the sequences.

## 3. How Many N-grams Are Generated?

For a token sequence of length `N`, the number of contiguous n-grams of size `n` is:

$$\max(0, N - n + 1)$$

For example, a sequence with 6 tokens contains 5 bigrams and 4 trigrams.

In [3]:
token_count = len(tokens)

print("Token count:", token_count)
print("Expected unigrams:", max(0, token_count - 1 + 1))
print("Expected bigrams:", max(0, token_count - 2 + 1))
print("Expected trigrams:", max(0, token_count - 3 + 1))

Token count: 8
Expected unigrams: 8
Expected bigrams: 7
Expected trigrams: 6


## 4. Count N-gram Frequencies

Use `Counter` to count how often each n-gram appears. Repeated phrases become visible in the frequency table.

In [4]:
from collections import Counter

sample_text = "natural language processing natural language understanding language processing"
sample_tokens = sample_text.split()

bigram_counts = Counter(ngrams(sample_tokens, 2))
trigram_counts = Counter(ngrams(sample_tokens, 3))

print("Most common bigrams:")
for gram, count in bigram_counts.most_common():
    print(gram, "->", count)

print("\nMost common trigrams:")
for gram, count in trigram_counts.most_common():
    print(gram, "->", count)

Most common bigrams:
('natural', 'language') -> 2
('language', 'processing') -> 2
('processing', 'natural') -> 1
('language', 'understanding') -> 1
('understanding', 'language') -> 1

Most common trigrams:
('natural', 'language', 'processing') -> 1
('language', 'processing', 'natural') -> 1
('processing', 'natural', 'language') -> 1
('natural', 'language', 'understanding') -> 1
('language', 'understanding', 'language') -> 1
('understanding', 'language', 'processing') -> 1


## 5. N-gram Frequency Distribution with NLTK

`FreqDist` can count n-gram tuples just as it counts individual tokens.

In [5]:
from nltk import FreqDist

bigram_fd = FreqDist(ngrams(sample_tokens, 2))

print("Number of distinct bigrams:", len(bigram_fd))
print("Most common bigrams:", bigram_fd.most_common(5))
print("Frequency of ('natural', 'language'):", bigram_fd[('natural', 'language')])

Number of distinct bigrams: 5
Most common bigrams: [(('natural', 'language'), 2), (('language', 'processing'), 2), (('processing', 'natural'), 1), (('language', 'understanding'), 1), (('understanding', 'language'), 1)]
Frequency of ('natural', 'language'): 2


## 6. Tokenization and Punctuation

Simple `split()` separates text on whitespace but leaves punctuation attached to words. In real NLP tasks, tokenization choices affect the n-grams you obtain.

In [6]:
import nltk
from nltk.tokenize import wordpunct_tokenize

punctuated_text = "NLP is useful, isn't it? NLP is powerful!"
punctuation_tokens = wordpunct_tokenize(punctuated_text.lower())
punctuation_bigrams = list(ngrams(punctuation_tokens, 2))

print("Tokens:", punctuation_tokens)
print("Bigrams:", punctuation_bigrams)

Tokens: ['nlp', 'is', 'useful', ',', 'isn', "'", 't', 'it', '?', 'nlp', 'is', 'powerful', '!']
Bigrams: [('nlp', 'is'), ('is', 'useful'), ('useful', ','), (',', 'isn'), ('isn', "'"), ("'", 't'), ('t', 'it'), ('it', '?'), ('?', 'nlp'), ('nlp', 'is'), ('is', 'powerful'), ('powerful', '!')]


Notice that punctuation may become separate tokens with `wordpunct_tokenize()`. This is useful for some tasks, but it may not be the desired behavior for every project. Choose a tokenizer that fits your task.

## 7. Compare Unigram and Bigram Counts

Unigram counts describe individual token frequency. Bigram counts preserve a little word-order information and can highlight common phrases.

In [7]:
from collections import Counter

review = "good product good service excellent product good quality"
review_tokens = review.split()

unigram_counts = Counter(review_tokens)
review_bigram_counts = Counter(ngrams(review_tokens, 2))

print("Unigrams:")
print(unigram_counts.most_common())

print("\nBigrams:")
print(review_bigram_counts.most_common())

Unigrams:
[('good', 3), ('product', 2), ('service', 1), ('excellent', 1), ('quality', 1)]

Bigrams:
[(('product', 'good'), 2), (('good', 'product'), 1), (('good', 'service'), 1), (('service', 'excellent'), 1), (('excellent', 'product'), 1), (('good', 'quality'), 1)]


## 8. Create a Reusable N-gram Function

The following function returns the most common n-grams for a chosen value of `n`. It validates the input so that `n` must be a positive integer.

In [8]:
from collections import Counter
from nltk.util import ngrams

def most_common_ngrams(tokens, n=2, top_k=10):
    """Return the top_k most frequent n-grams from a token sequence."""
    if not isinstance(n, int) or n < 1:
        raise ValueError("n must be a positive integer")
    if not isinstance(top_k, int) or top_k < 1:
        raise ValueError("top_k must be a positive integer")

    counts = Counter(ngrams(tokens, n))
    return counts.most_common(top_k)

example_tokens = "data science uses data analysis and data visualization".split()
print(most_common_ngrams(example_tokens, n=1, top_k=5))
print(most_common_ngrams(example_tokens, n=2, top_k=5))

[(('data',), 3), (('science',), 1), (('uses',), 1), (('analysis',), 1), (('and',), 1)]
[(('data', 'science'), 1), (('science', 'uses'), 1), (('uses', 'data'), 1), (('data', 'analysis'), 1), (('analysis', 'and'), 1)]


## 9. Applications of N-grams

N-grams have been used in:
- **Language modeling:** estimating likely next tokens from preceding tokens.
- **Search and text matching:** finding common phrases.
- **Text classification:** using word sequences as features.
- **Spelling correction:** comparing possible word sequences with observed patterns.
- **Corpus analysis:** identifying repeated expressions and collocations.

N-grams capture local context, but they do not directly model long-distance relationships.

## 10. Limitations

- **Data sparsity:** many possible n-grams may never appear in a training corpus.
- **Memory usage:** larger n values can produce many distinct sequences.
- **Limited context:** a trigram only captures three consecutive tokens.
- **Domain dependence:** common phrases can differ between domains.
- **Preprocessing sensitivity:** tokenization, casing, and punctuation change the counts.

Increasing `n` gives more local context but often makes sparsity worse.

## 11. Practical Example: Phrase Analysis

This example analyzes a small collection of product reviews and extracts frequently occurring bigrams. The same workflow can be adapted to larger text datasets.

The example keeps preprocessing simple so the relationship between tokenization, n-gram generation, and frequency counting remains clear.

In [9]:
from collections import Counter
from nltk.util import ngrams

reviews = [
    "The product is easy to use and the quality is excellent.",
    "Excellent product with excellent quality.",
    "The service is easy to use.",
    "Good quality product and helpful customer service.",
]

review_tokens = []
for review in reviews:
    tokens_in_review = review.lower().replace(".", "").split()
    review_tokens.extend(tokens_in_review)

bigram_counts = Counter(ngrams(review_tokens, 2))

print("Top 10 bigrams across the review collection:")
for bigram, count in bigram_counts.most_common(10):
    print(f"{' '.join(bigram):<25} {count}")

Top 10 bigrams across the review collection:
is easy                   2
easy to                   2
to use                    2
the product               1
product is                1
use and                   1
and the                   1
the quality               1
quality is                1
is excellent              1


## Summary

- An n-gram is a sequence of `n` consecutive tokens.
- `nltk.util.ngrams()` generates n-gram tuples.
- `Counter` and `nltk.FreqDist` count their frequencies.
- Tokenization and normalization influence the results.
- N-grams are useful for local context but can suffer from data sparsity.